# Step 2 v3: Feature table version 3 + fair comparison of feature sets
**Owner:** Fenny (features), with Jian. **Needs:** `features_v3.py` in this folder, the step 1 files in `clean/`.

This notebook:
1. builds v3 features for both cities with ONE function (so features mean the same in both),
2. checks, WITHOUT any Amsterdam labels, which features shift between cities,
3. runs a fair comparison of v1, v2, v3 and a "stable" v3 subset: same model, same folds, same seeds.

It never overwrites v1 or v2 files. It writes `features_madrid_v3.parquet`,
`features_amsterdam_v3.parquet`, `feature_cols_v3.json`, `feature_sets_v3.json`, `feature_comparison_v3.csv`.

In [1]:
import json, time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import features_v3 as fv3          # our feature code (read features_v3.py, every step is commented)

SEED = 42
CLEAN = Path("clean")
SHIFT_KS_LIMIT = 0.30              # "stable" subset drops features whose distribution moves more than this
print("ready")

ready


## 1. Build v3 for both cities (same function)
132 features = 11 channels (6 bands + NDVI, NDBI, MNDWI, NDSWIR, BSI) x 12 statistics.
Level statistics use gap-filled years; change statistics use observed years only.

In [2]:
t = time.time()
feat_mad, COLS_V3 = fv3.build_features(pd.read_parquet(CLEAN / "clean_madrid.parquet"))
feat_ams, _ = fv3.build_features(pd.read_parquet(CLEAN / "clean_amsterdam.parquet"))
print(f"done in {time.time()-t:.0f}s")

# Checks: right number of pixels, same class counts as step 1, no missing values, no forbidden columns
assert len(feat_mad) == 76263 and len(feat_ams) == 25992, "pixel count changed"
assert feat_mad[COLS_V3].isna().sum().sum() == 0 and feat_ams[COLS_V3].isna().sum().sum() == 0
assert not set(COLS_V3) & fv3.FORBIDDEN
print("Madrid classes:", feat_mad["age_class"].value_counts().sort_index().to_dict())
print("Amsterdam classes:", feat_ams["age_class"].value_counts().sort_index().to_dict())

feat_mad.to_parquet("features_madrid_v3.parquet", index=False)
feat_ams.to_parquet("features_amsterdam_v3.parquet", index=False)
json.dump(COLS_V3, open("feature_cols_v3.json", "w"))
print("saved v3 files")

built 76,263 pixels x 132 features (0 NaN values filled with the city median)
built 25,992 pixels x 132 features (0 NaN values filled with the city median)
done in 75s
Madrid classes: {1: 13794, 2: 27152, 3: 18672, 4: 16645}
Amsterdam classes: {1: 7568, 2: 8802, 3: 6745, 4: 2877}
saved v3 files


## 2. Which features move between cities? (no labels used)
KS statistic: 0 = same distribution in both cities, 1 = completely different.
A big shift is a WARNING, not proof: part of it is real (Amsterdam has fewer class 4 buildings).
That is why we do not just drop them, we TEST the stable subset in step 3.

In [3]:
shift = fv3.shift_report(feat_mad, feat_ams, COLS_V3)
shift.to_csv("feature_shift_v3.csv", index=False)
print("Most stable:\n", shift.head(10).round(3).to_string(index=False))
print("\nMost shifted:\n", shift.tail(10).round(3).to_string(index=False))
STABLE_V3 = shift.loc[shift["ks"] <= SHIFT_KS_LIMIT, "feature"].tolist()
print(f"\nstable subset: {len(STABLE_V3)} of {len(COLS_V3)} features (KS <= {SHIFT_KS_LIMIT})")

Most stable:
                feature    ks  median_shift_sd
         Blue_max_jump 0.054            0.070
          NIR_late_std 0.076            0.129
          Blue_chg_std 0.101            0.147
SWIR2_year_of_max_jump 0.128            0.075
  NIR_late_minus_early 0.134            0.145
          Red_chg_mean 0.135            0.055
        SWIR1_max_jump 0.136            0.209
      MNDWI_early_mean 0.143            0.359
           MNDWI_slope 0.167            0.275
 NDVI_late_minus_early 0.168            0.197

Most shifted:
          feature    ks  median_shift_sd
       Blue_mean 0.721            1.322
 Blue_early_mean 0.740            1.604
   Red_late_mean 0.746            1.523
      SWIR2_mean 0.747            1.738
      Green_mean 0.758            1.454
Green_early_mean 0.769            1.727
SWIR2_early_mean 0.781            1.997
    NDVI_chg_std 0.796            2.712
        Red_mean 0.817            1.926
  Red_early_mean 0.838            2.108

stable subset: 45 of 13

## 3. Fair comparison of feature sets
Same for every set: Random Forest (300 trees, balanced classes, seed 42), the same 5 Madrid folds,
the same 10 Amsterdam support samples. Only the features change.
* Madrid: 5-fold cross-validated macro F1 (scaler + model fitted inside each fold).
* Amsterdam: official prototype method at n = 5 and n = 25 per class. Scaler fitted on Madrid only.
  Support pixels are never scored.

In [4]:
def madrid_cv(X, y):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    scores = []
    for tr, va in skf.split(X, y):
        sc = StandardScaler().fit(X[tr])                      # fitted on training fold only
        rf = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                    random_state=SEED, n_jobs=-1)
        rf.fit(sc.transform(X[tr]), y[tr])
        scores.append(f1_score(y[va], rf.predict(sc.transform(X[va])), average="macro"))
    return np.mean(scores), np.std(scores)


def amsterdam_proto(Xm, Xa, ya, n, seeds=range(SEED, SEED + 10)):
    sc = StandardScaler().fit(Xm)                              # Madrid only, no Amsterdam labels
    Z = sc.transform(Xa)
    scores = []
    for s in seeds:
        rng = np.random.default_rng(s)
        sup = np.concatenate([rng.choice(np.where(ya == c)[0], n, replace=False) for c in (1, 2, 3, 4)])
        qry = np.setdiff1d(np.arange(len(ya)), sup)
        protos = np.stack([Z[sup][ya[sup] == c].mean(axis=0) for c in (1, 2, 3, 4)])
        d = ((Z[qry][:, None, :] - protos[None]) ** 2).sum(axis=2)
        scores.append(f1_score(ya[qry], np.array([1, 2, 3, 4])[d.argmin(axis=1)], average="macro"))
    return np.mean(scores), np.std(scores)


def load_set(name):
    """Return (Madrid table, Amsterdam table, feature list) for each feature set."""
    if name == "v1":
        return (pd.read_parquet("features_madrid.parquet"), pd.read_parquet("features_amsterdam.parquet"),
                json.load(open("feature_cols.json")))
    if name == "v2_all":
        m, a = pd.read_parquet("features_madrid_v2.parquet"), pd.read_parquet("features_amsterdam_v2.parquet")
        cols = [c for c in m.columns if c not in fv3.FORBIDDEN]   # all v2 features, no leaked list
        return m, a, cols
    if name == "v3_all":
        return feat_mad, feat_ams, COLS_V3
    if name == "v3_stable":
        return feat_mad, feat_ams, STABLE_V3


rows = []
for name in ["v1", "v2_all", "v3_all", "v3_stable"]:
    t = time.time()
    m, a, cols = load_set(name)
    a = a.set_index(["px_key", "py_key"]).loc[feat_ams.set_index(["px_key", "py_key"]).index].reset_index()
    Xm, ym = m[cols].to_numpy(float), m["age_class"].to_numpy(int)
    Xa, ya = a[cols].to_numpy(float), a["age_class"].to_numpy(int)
    mad_mean, mad_std = madrid_cv(Xm, ym)
    a5, a5s = amsterdam_proto(Xm, Xa, ya, 5)
    a25, a25s = amsterdam_proto(Xm, Xa, ya, 25)
    rows.append({"feature_set": name, "n_features": len(cols),
                 "madrid_f1": mad_mean, "madrid_std": mad_std,
                 "ams_n5_f1": a5, "ams_n5_std": a5s, "ams_n25_f1": a25, "ams_n25_std": a25s})
    print(f"{name:10s} {len(cols):4d} feats | Madrid {mad_mean:.4f}+/-{mad_std:.4f} | "
          f"Ams n=5 {a5:.4f} | n=25 {a25:.4f}  ({time.time()-t:.0f}s)")

comparison = pd.DataFrame(rows).round(4)
comparison.to_csv("feature_comparison_v3.csv", index=False)
comparison

v1           30 feats | Madrid 0.6099+/-0.0056 | Ams n=5 0.4882 | n=25 0.5119  (81s)
v2_all       46 feats | Madrid 0.6308+/-0.0052 | Ams n=5 0.4926 | n=25 0.5306  (51s)
v3_all      132 feats | Madrid 0.6581+/-0.0039 | Ams n=5 0.5040 | n=25 0.5547  (104s)
v3_stable    45 feats | Madrid 0.6113+/-0.0050 | Ams n=5 0.4863 | n=25 0.5248  (87s)


,feature_set,n_features,madrid_f1,madrid_std,ams_n5_f1,ams_n5_std,ams_n25_f1,ams_n25_std
0,v1,30,0.6099,0.0056,0.4882,0.0447,0.5119,0.0286
1,v2_all,46,0.6308,0.0052,0.4926,0.0406,0.5306,0.0144
2,v3_all,132,0.6581,0.0039,0.5040,0.0454,0.5547,0.0129
3,v3_stable,45,0.6113,0.0050,0.4863,0.0340,0.5248,0.0122


## 4. Decision rule (agree on this BEFORE looking at the numbers)
* The winner is the set with the best **Amsterdam n=25** score (the low-data prize), as long as its
  Madrid score is not more than 0.01 below the best Madrid score.
* If two sets are within one standard deviation of each other, take the one with FEWER features
  (simpler, more likely to work in a third city).
* The winning list is saved to `feature_sets_v3.json` and handed to Silvio, Antonio and Anselmo.

Note: the official reference used 60 features and scored Madrid 0.618, Amsterdam n=5 0.544, n=25 0.599.

In [5]:
best_mad = comparison["madrid_f1"].max()
ok = comparison[comparison["madrid_f1"] >= best_mad - 0.01].sort_values("ams_n25_f1", ascending=False)
top = ok.iloc[0]
close = ok[ok["ams_n25_f1"] >= top["ams_n25_f1"] - top["ams_n25_std"]].sort_values("n_features")
winner = close.iloc[0]["feature_set"]
print("WINNER:", winner)
json.dump({"winner": winner, "v3_all": COLS_V3, "v3_stable": STABLE_V3},
          open("feature_sets_v3.json", "w"), indent=1)
print("saved feature_sets_v3.json")

WINNER: v3_all
saved feature_sets_v3.json
